# 3DOF RL Control

This notebook turns the 3DOF training-rig simulator into a reinforcement-learning testbed.

Main capabilities in this version:
- one **hyperparameter cell** for task, reward, saving, visualization, and device selection
- clean forward-dynamics wrapper using the same symbolic plant as the simulation notebook
- support for **fixed setpoints**, **fixed trajectories**, **random setpoints**, and **random trajectories**
- training and comparison of multiple continuous-control RL algorithms
- reward-vs-iteration plots
- rollout saving during training so you can animate **any saved iteration**, not only the final one
- animation with a **target point marker** for the UAV body based on the desired joint angles

Recommended kernel/interpreter:
- use the local virtual environment: `.venv_rl/bin/python`


In [ ]:
import os
import json
import math
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import torch

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO, SAC, TD3
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

from geometry_param import get_params, get_quad_params


## Hyperparameters

This is the main cell to edit.

You can control here:
- CPU/GPU selection
- which RL algorithms to train
- simulation timing
- whether initial conditions are randomized
- whether goals are fixed or randomized
- reward weights
- save cadence for checkpoints and rollouts
- which saved iteration to animate later


In [ ]:
# ============================================================
# DEVICE SELECTION
# ============================================================
# Choose one of: "cpu", "gpu", "auto"
DEVICE_CHOICE = "auto"

if DEVICE_CHOICE == "cpu":
    RL_DEVICE = "cpu"
elif DEVICE_CHOICE == "gpu":
    RL_DEVICE = "cuda"
elif DEVICE_CHOICE == "auto":
    RL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    raise ValueError(f"Unknown DEVICE_CHOICE: {DEVICE_CHOICE}")

USE_CUDA = RL_DEVICE == "cuda" and torch.cuda.is_available()
print("Requested device mode:", DEVICE_CHOICE)
print("Resolved training device:", RL_DEVICE)
print("torch.cuda.is_available():", torch.cuda.is_available())
if USE_CUDA:
    print("CUDA device name:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# FILES AND DIRECTORIES
# ============================================================
SYMBOLIC_FILE = Path("cache/3DOF_symbolic_data_04052026_0057.json")
RESULTS_DIR = Path("results")
RL_DIR = RESULTS_DIR / "rl"
MODEL_DIR = RL_DIR / "models"
CHECKPOINT_DIR = RL_DIR / "checkpoints"
ROLLOUT_DIR = RL_DIR / "rollouts"
LOG_DIR = RL_DIR / "logs"

for directory in [RESULTS_DIR, RL_DIR, MODEL_DIR, CHECKPOINT_DIR, ROLLOUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ============================================================
# GLOBAL FLAGS
# ============================================================
SEED = 7
SAVE_CHECKPOINTS = True
SAVE_PERIODIC_ROLLOUTS = True
SAVE_FINAL_ROLLOUT = True
SAVE_TRAINING_HISTORY = True
RUN_ENV_CHECK = True

# ============================================================
# DEVICE SELECTION
# ============================================================
# Choose one of: "cpu", "gpu", "auto"
DEVICE_CHOICE = "auto"

if DEVICE_CHOICE == "cpu":
    RL_DEVICE = "cpu"
elif DEVICE_CHOICE == "gpu":
    RL_DEVICE = "cuda"
elif DEVICE_CHOICE == "auto":
    RL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    raise ValueError(f"Unknown DEVICE_CHOICE: {DEVICE_CHOICE}")

USE_CUDA = RL_DEVICE == "cuda" and torch.cuda.is_available()
print("Requested device mode:", DEVICE_CHOICE)
print("Resolved training device:", RL_DEVICE)
print("torch.cuda.is_available():", torch.cuda.is_available())
if USE_CUDA:
    print("CUDA device name:", torch.cuda.get_device_name(0))

# ============================================================
# TASK DEFINITION
# ============================================================
GOAL_MODE = "fixed_point"

FIXED_DESIRED_STATE = np.array([
    np.deg2rad(-45.0),
    np.deg2rad(60.0),
    np.deg2rad(0.0),
    0.0,
    0.0,
    0.0,
], dtype=float)

FIXED_TRAJ_BIAS = np.array([
    np.deg2rad(0.0),
    np.deg2rad(5.0),
    np.deg2rad(0.0),
], dtype=float)
FIXED_TRAJ_AMPLITUDE = np.array([
    np.deg2rad(5.0),
    np.deg2rad(4.0),
    np.deg2rad(5.0),
], dtype=float)
FIXED_TRAJ_OMEGA = np.array([0.6, 0.4, 0.5], dtype=float)
FIXED_TRAJ_PHASE = np.array([0.0, 0.2, -0.1], dtype=float)

RANDOM_POINT_ANGLE_LOW = np.deg2rad(np.array([-20.0, -20.0, -20.0]))
RANDOM_POINT_ANGLE_HIGH = np.deg2rad(np.array([20.0, 20.0, 20.0]))
RANDOM_POINT_RATE_LOW = np.deg2rad(np.array([-20.0, -20.0, -20.0]))
RANDOM_POINT_RATE_HIGH = np.deg2rad(np.array([20.0, 20.0, 20.0]))

RANDOM_TRAJ_BIAS_LOW = np.deg2rad(np.array([-15.0, -15.0, -15.0]))
RANDOM_TRAJ_BIAS_HIGH = np.deg2rad(np.array([15.0, 15.0, 15.0]))
RANDOM_TRAJ_AMP_LOW = np.deg2rad(np.array([2.0, 2.0, 2.0]))
RANDOM_TRAJ_AMP_HIGH = np.deg2rad(np.array([8.0, 8.0, 8.0]))
RANDOM_TRAJ_OMEGA_LOW = np.array([0.2, 0.2, 0.2], dtype=float)
RANDOM_TRAJ_OMEGA_HIGH = np.array([1.0, 1.0, 1.0], dtype=float)

# ============================================================
# SIMULATION TIMING
# ============================================================
SIM_T_START = 0.0
SIM_T_END = 60.0
DT = 0.02
EPISODE_STEPS = int(round((SIM_T_END - SIM_T_START) / DT))
print("Episode steps:", EPISODE_STEPS)
print("Episode duration [s]:", EPISODE_STEPS * DT)

# ============================================================
# INITIAL CONDITION RANDOMIZATION
# ============================================================
RANDOMIZE_INITIAL_STATE = True
INITIAL_STATE_MEAN = np.array([
    np.deg2rad(0.0),
    np.deg2rad(0.0),
    np.deg2rad(0.0),
    0.0,
    0.0,
    0.0,
], dtype=float)
INITIAL_STATE_HALF_WIDTH = np.array([
    np.deg2rad(5.0),
    np.deg2rad(5.0),
    np.deg2rad(5.0),
    np.deg2rad(10.0),
    np.deg2rad(10.0),
    np.deg2rad(10.0),
], dtype=float)

# ============================================================
# ACTION / QUAD MODEL HYPERPARAMETERS
# ============================================================
HOVER_RPM = 20e3
RPM_DELTA_LIMIT = 7000.0
MAX_RPM = 50e3
KF = 2.0e-8
KM = 2.0e-10

# ============================================================
# REWARD / SUCCESS HYPERPARAMETERS
# ============================================================
REWARD_WEIGHTS = {
    "angle_error": 20.0,
    "rate_error": 2.0,
    "action_effort": 0.02,
    "action_delta": 0.01,
}
SUCCESS_ANGLE_NORM_THRESHOLD = np.deg2rad(2.0)
SUCCESS_RATE_NORM_THRESHOLD = np.deg2rad(5.0)
SUCCESS_BONUS = 5.0
SUCCESS_HOLD_BONUS_PER_STEP = 2.0
SUCCESS_HOLD_TIME_SEC = 10.0
TERMINATE_ON_SUCCESS = False
TERMINAL_FAILURE_PENALTY = 50.0

# ============================================================
# RL ALGORITHMS TO TRAIN
# ============================================================
ALGORITHMS_TO_TRAIN = ["PPO", "SAC", "TD3"]

TRAIN_TIMESTEPS = {
    "PPO": 600_000,
    "SAC": 600_000,
    "TD3": 600_000,
}

PPO_KWARGS = {
    "learning_rate": 3e-4,
    "n_steps": 1024,
    "batch_size": 256,
    "gamma": 0.995,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
    "ent_coef": 0.0,
    "verbose": 1,
    "device": RL_DEVICE,
}

SAC_KWARGS = {
    "learning_rate": 3e-4,
    "batch_size": 256,
    "buffer_size": 100_000,
    "learning_starts": 2_000,
    "gamma": 0.995,
    "tau": 0.005,
    "train_freq": 1,
    "gradient_steps": 1,
    "verbose": 1,
    "device": RL_DEVICE,
}

TD3_KWARGS = {
    "learning_rate": 1e-3,
    "batch_size": 256,
    "buffer_size": 100_000,
    "learning_starts": 2_000,
    "gamma": 0.995,
    "tau": 0.005,
    "train_freq": 1,
    "gradient_steps": 1,
    "verbose": 1,
    "device": RL_DEVICE,
}

# ============================================================
# SAVE / EVAL / VISUALIZATION CONTROL
# ============================================================
CHECKPOINT_EVERY = 5_000
EVAL_EVERY = 5_000
ROLLOUT_SAVE_EVERY = 10_000

VISUALIZE_ALGO = "PPO"
VISUALIZE_ROLLOUT_SELECTOR = "final"



## Load Symbolic Plant

We reuse the same symbolic model as the forward simulation notebook. The RL environment only changes the control policy and goal logic.


In [ ]:
# ============================================================
# DESERIALIZATION HELPERS
# ============================================================
SYMPY_LOCALS = {name: getattr(sp, name) for name in dir(sp)}
SYMPY_LOCALS.update({"Derivative": sp.Derivative})


def deserialize_fast(obj):
    if isinstance(obj, dict) and "__sympy__" in obj:
        return eval(obj["expr"], SYMPY_LOCALS)
    if isinstance(obj, dict):
        return {k: deserialize_fast(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [deserialize_fast(v) for v in obj]
    return obj


with open(SYMBOLIC_FILE, "r") as f:
    raw = json.load(f)

data = deserialize_fast(raw)

M_sym = data["M"]
h_sym = data["h"]
Q_sym = data["Q"]

n = data["n"]
nd = data["nd"]
params_sym = data["params"]

state_syms = list(n) + list(nd)
param_syms = list(params_sym.values())
tau_syms = sp.symbols("tau1 tau2 tau3")
F_syms = sp.symbols("Fx Fy Fz")

dyn_args = (
    *state_syms,
    *param_syms,
    *tau_syms,
    *F_syms,
)

print("Loaded symbolic model from", SYMBOLIC_FILE)


In [ ]:
# Lambdify once so all training environments reuse the same fast NumPy callables.
print("Lambdifying symbolic dynamics...")
M_func = sp.lambdify(dyn_args, M_sym, "numpy")
h_func = sp.lambdify(dyn_args, h_sym, "numpy")
Q_func = sp.lambdify(dyn_args, Q_sym, "numpy")
print("Done.")


## Forward Dynamics Wrapper

This class turns rotor RPM commands into body wrench, then into state derivatives through the symbolic plant.


In [ ]:
@dataclass
class QuadConfig:
    quad_l: float
    hover_rpm: float
    rpm_delta_limit: float
    max_rpm: float
    kf: float
    km: float


class ThreeDOFDynamicsModel:
    def __init__(self, params_num, quad_meta, quad_config, dt):
        self.params_num = dict(params_num)
        self.quad_meta = dict(quad_meta)
        self.quad_config = quad_config
        self.dt = float(dt)
        self.param_vals = list(self.params_num.values())

    def rpm_from_action(self, action):
        # Action is normalized in [-1, 1]. Convert it to RPM around hover.
        action = np.asarray(action, dtype=float)
        rpm = self.quad_config.hover_rpm + self.quad_config.rpm_delta_limit * action
        return np.clip(rpm, 0.0, self.quad_config.max_rpm)

    def quad_wrench_from_rpm(self, rpm):
        # Net quad force is constrained to body +Z only.
        rpm = np.asarray(rpm, dtype=float)
        omega = rpm * (2.0 * np.pi / 60.0)
        w1, w2, w3, w4 = omega

        f1 = self.quad_config.kf * w1**2
        f2 = self.quad_config.kf * w2**2
        f3 = self.quad_config.kf * w3**2
        f4 = self.quad_config.kf * w4**2

        F_B = np.array([0.0, 0.0, f1 + f2 + f3 + f4], dtype=float)
        tau_B = np.array([
            self.quad_config.quad_l * (f2 - f4),
            self.quad_config.quad_l * (f3 - f1),
            self.quad_config.km * (w1**2 - w2**2 + w3**2 - w4**2),
        ], dtype=float)
        return F_B, tau_B

    def state_derivative_from_rpm(self, state, rpm):
        # Evaluate M(q), h(q, qdot), and Q(q, qdot, inputs), then solve for qddot.
        state = np.asarray(state, dtype=float)
        n_val = state[:3]
        nd_val = state[3:]
        F_A, tau_A = self.quad_wrench_from_rpm(rpm)

        args = (
            *n_val,
            *nd_val,
            *self.param_vals,
            *tau_A,
            *F_A,
        )

        M = np.array(M_func(*args), dtype=float)
        h = np.array(h_func(*args), dtype=float).reshape(3)
        Q = np.array(Q_func(*args), dtype=float).reshape(3)

        try:
            ndd = np.linalg.solve(M, Q - h)
        except np.linalg.LinAlgError:
            # Least-squares fallback avoids hard crashes in ill-conditioned states.
            ndd = np.linalg.lstsq(M, Q - h, rcond=None)[0]

        return np.hstack([nd_val, ndd]), F_A, tau_A

    def rk4_step_from_action(self, state, action):
        # Use RK4 instead of Euler so policy training sees a cleaner plant.
        rpm = self.rpm_from_action(action)

        def f(x):
            dx, _, _ = self.state_derivative_from_rpm(x, rpm)
            return dx

        k1 = f(state)
        k2 = f(state + 0.5 * self.dt * k1)
        k3 = f(state + 0.5 * self.dt * k2)
        k4 = f(state + self.dt * k3)
        next_state = state + (self.dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

        # Re-evaluate the wrench at the returned state only for logging clarity.
        _, F_A, tau_A = self.state_derivative_from_rpm(next_state, rpm)
        return next_state, rpm, F_A, tau_A


## Desired-State Manager

This helper lets one environment handle fixed goals, fixed trajectories, randomized setpoints, and randomized trajectories.

That also answers one of your main design questions:
- **yes**, you can train one policy for a family of states or trajectories
- the standard way is to make the policy **goal-conditioned** and randomize goals during training
- you do **not** need continual learning for the first version; goal randomization is the cleaner first step


In [ ]:
class DesiredStateManager:
    def __init__(self, goal_mode, fixed_state, fixed_traj_params, rng):
        self.goal_mode = goal_mode
        self.fixed_state = np.asarray(fixed_state, dtype=float)
        self.fixed_traj_params = fixed_traj_params
        self.rng = rng
        self.episode_spec = None

    def _sample_random_point(self):
        angles = self.rng.uniform(RANDOM_POINT_ANGLE_LOW, RANDOM_POINT_ANGLE_HIGH)
        rates = self.rng.uniform(RANDOM_POINT_RATE_LOW, RANDOM_POINT_RATE_HIGH)
        return np.hstack([angles, rates])

    def _sample_random_traj_spec(self):
        return {
            "bias": self.rng.uniform(RANDOM_TRAJ_BIAS_LOW, RANDOM_TRAJ_BIAS_HIGH),
            "amplitude": self.rng.uniform(RANDOM_TRAJ_AMP_LOW, RANDOM_TRAJ_AMP_HIGH),
            "omega": self.rng.uniform(RANDOM_TRAJ_OMEGA_LOW, RANDOM_TRAJ_OMEGA_HIGH),
            "phase": self.rng.uniform(-np.pi, np.pi, size=3),
        }

    def reset_episode(self):
        if self.goal_mode == "fixed_point":
            self.episode_spec = {"type": "point", "state": self.fixed_state.copy()}
        elif self.goal_mode == "fixed_trajectory":
            self.episode_spec = {"type": "trajectory", **self.fixed_traj_params}
        elif self.goal_mode == "random_point":
            self.episode_spec = {"type": "point", "state": self._sample_random_point()}
        elif self.goal_mode == "random_trajectory":
            self.episode_spec = {"type": "trajectory", **self._sample_random_traj_spec()}
        else:
            raise ValueError(f"Unknown goal mode: {self.goal_mode}")

    def desired_state(self, t):
        if self.episode_spec is None:
            raise RuntimeError("reset_episode() must be called before desired_state().")

        if self.episode_spec["type"] == "point":
            return self.episode_spec["state"].copy()

        bias = np.asarray(self.episode_spec["bias"], dtype=float)
        amplitude = np.asarray(self.episode_spec["amplitude"], dtype=float)
        omega = np.asarray(self.episode_spec["omega"], dtype=float)
        phase = np.asarray(self.episode_spec["phase"], dtype=float)

        angles = bias + amplitude * np.sin(omega * t + phase)
        rates = amplitude * omega * np.cos(omega * t + phase)
        return np.hstack([angles, rates])


## RL Environment

Observation contents:
- current state
- current desired state
- tracking error

Including the desired state explicitly is important if you want one policy to generalize across many targets.


In [ ]:
class ThreeDOFQuadRLEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, seed=None):
        super().__init__()

        self.params_num = get_params()
        self.quad_meta = get_quad_params()
        self.quad_config = QuadConfig(
            quad_l=float(self.quad_meta["quad_l"]),
            hover_rpm=HOVER_RPM,
            rpm_delta_limit=RPM_DELTA_LIMIT,
            max_rpm=MAX_RPM,
            kf=KF,
            km=KM,
        )
        self.model = ThreeDOFDynamicsModel(self.params_num, self.quad_meta, self.quad_config, DT)

        self.sim_t_start = float(SIM_T_START)
        self.sim_t_end = float(SIM_T_END)
        self.dt = float(DT)
        self.episode_steps = int(EPISODE_STEPS)
        self.success_hold_steps_required = max(1, int(round(SUCCESS_HOLD_TIME_SEC / self.dt)))

        self.np_random = None
        self.goal_manager = None
        self.current_time = self.sim_t_start
        self.state = np.zeros(6, dtype=float)
        self.last_action = np.zeros(4, dtype=float)
        self.steps = 0
        self.hold_steps = 0
        self.success_achieved = False
        self.termination_reason = ""

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(18,), dtype=np.float32)

        self.reset(seed=seed)

    def _get_rng(self, seed=None):
        self.np_random, _ = gym.utils.seeding.np_random(seed)
        return self.np_random

    def _build_goal_manager(self):
        fixed_traj_params = {
            "bias": FIXED_TRAJ_BIAS,
            "amplitude": FIXED_TRAJ_AMPLITUDE,
            "omega": FIXED_TRAJ_OMEGA,
            "phase": FIXED_TRAJ_PHASE,
        }
        self.goal_manager = DesiredStateManager(
            goal_mode=GOAL_MODE,
            fixed_state=FIXED_DESIRED_STATE,
            fixed_traj_params=fixed_traj_params,
            rng=self.np_random,
        )
        self.goal_manager.reset_episode()

    def current_desired_state(self):
        return self.goal_manager.desired_state(self.current_time)

    def current_error(self):
        return self.current_desired_state() - self.state

    def _get_obs(self):
        desired = self.current_desired_state()
        error = desired - self.state
        obs = np.hstack([self.state, desired, error])
        return obs.astype(np.float32)

    def _sample_initial_state(self):
        if RANDOMIZE_INITIAL_STATE:
            rand = self.np_random.uniform(low=-1.0, high=1.0, size=6)
            return INITIAL_STATE_MEAN + rand * INITIAL_STATE_HALF_WIDTH
        return INITIAL_STATE_MEAN.copy()

    def _tracking_errors(self):
        desired = self.current_desired_state()
        angle_error = desired[:3] - self.state[:3]
        rate_error = desired[3:] - self.state[3:]
        return angle_error, rate_error

    def _within_success_tube(self):
        angle_error, rate_error = self._tracking_errors()
        return (
            np.linalg.norm(angle_error) < SUCCESS_ANGLE_NORM_THRESHOLD
            and np.linalg.norm(rate_error) < SUCCESS_RATE_NORM_THRESHOLD
        )

    def _update_success_hold(self):
        if self._within_success_tube():
            self.hold_steps += 1
        else:
            self.hold_steps = 0
        self.success_achieved = self.hold_steps >= self.success_hold_steps_required

    def _reward(self, action):
        angle_error, rate_error = self._tracking_errors()
        action = np.asarray(action, dtype=float)
        action_delta = action - self.last_action

        reward = 0.0
        reward -= REWARD_WEIGHTS["angle_error"] * float(angle_error @ angle_error)
        reward -= REWARD_WEIGHTS["rate_error"] * float(rate_error @ rate_error)
        reward -= REWARD_WEIGHTS["action_effort"] * float(action @ action)
        reward -= REWARD_WEIGHTS["action_delta"] * float(action_delta @ action_delta)

        if self._within_success_tube():
            reward += SUCCESS_BONUS
            reward += SUCCESS_HOLD_BONUS_PER_STEP * min(self.hold_steps, self.success_hold_steps_required)

        return reward

    def _failure_terminated(self):
        if not np.all(np.isfinite(self.state)):
            self.termination_reason = "non_finite_state"
            return True
        if np.max(np.abs(self.state[:3])) > np.deg2rad(80.0):
            self.termination_reason = "angle_limit"
            return True
        if np.max(np.abs(self.state[3:])) > np.deg2rad(400.0):
            self.termination_reason = "rate_limit"
            return True
        return False

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._get_rng(seed)
        self._build_goal_manager()
        self.current_time = self.sim_t_start
        self.state = self._sample_initial_state()
        self.last_action = np.zeros(4, dtype=float)
        self.steps = 0
        self.hold_steps = 0
        self.success_achieved = False
        self.termination_reason = ""

        info = {
            "desired_state": self.current_desired_state().copy(),
            "goal_mode": GOAL_MODE,
            "hold_time_sec": 0.0,
        }
        return self._get_obs(), info

    def step(self, action):
        action = np.asarray(action, dtype=float)
        action = np.clip(action, self.action_space.low, self.action_space.high)

        next_state, rpm, F_A, tau_A = self.model.rk4_step_from_action(self.state, action)
        self.state = next_state
        self.current_time += self.dt
        self.steps += 1

        self._update_success_hold()
        reward = self._reward(action)

        terminated = self._failure_terminated()
        if terminated:
            reward -= TERMINAL_FAILURE_PENALTY

        if (not terminated) and self.success_achieved and TERMINATE_ON_SUCCESS:
            terminated = True
            self.termination_reason = "success_hold_complete"

        truncated = self.steps >= self.episode_steps
        if truncated and not terminated:
            self.termination_reason = "time_limit"

        info = {
            "time": self.current_time,
            "desired_state": self.current_desired_state().copy(),
            "rpm": rpm.copy(),
            "F_A": F_A.copy(),
            "tau_A": tau_A.copy(),
            "hold_steps": self.hold_steps,
            "hold_time_sec": self.hold_steps * self.dt,
            "success_achieved": self.success_achieved,
            "termination_reason": self.termination_reason,
        }

        self.last_action = action.copy()
        return self._get_obs(), reward, terminated, truncated, info



In [ ]:
if RUN_ENV_CHECK:
    env_smoke = ThreeDOFQuadRLEnv(seed=SEED)
    check_env(env_smoke, warn=True)
    print("Environment check passed.")


## Rollout Helpers And Training Callbacks

We log both:
- episode reward vs training timestep
- evaluation reward vs training timestep

That gives the reward-vs-iteration plots you asked for.


In [ ]:
def run_policy_rollout(model, env, max_steps=None, deterministic=True):
    obs, info = env.reset(seed=SEED)
    max_steps = env.episode_steps if max_steps is None else int(max_steps)

    t_hist = []
    x_hist = []
    desired_hist = []
    rpm_hist = []
    F_hist = []
    tau_hist = []
    action_hist = []
    reward_hist = []
    hold_time_hist = []

    final_terminated = False
    final_truncated = False
    final_reason = ""

    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        t_hist.append(env.current_time)
        x_hist.append(env.state.copy())
        desired_hist.append(info["desired_state"].copy())
        rpm_hist.append(info["rpm"].copy())
        F_hist.append(info["F_A"].copy())
        tau_hist.append(info["tau_A"].copy())
        action_hist.append(np.asarray(action, dtype=float).copy())
        reward_hist.append(float(reward))
        hold_time_hist.append(float(info["hold_time_sec"]))

        final_terminated = terminated
        final_truncated = truncated
        final_reason = info.get("termination_reason", "")

        if terminated or truncated:
            break

    x_hist = np.array(x_hist, dtype=float)
    desired_hist = np.array(desired_hist, dtype=float)

    rollout = {
        "t": np.array(t_hist, dtype=float),
        "n": x_hist[:, :3],
        "nd": x_hist[:, 3:],
        "desired_state_history": desired_hist,
        "rpm": np.array(rpm_hist, dtype=float),
        "F": np.array(F_hist, dtype=float),
        "tau": np.array(tau_hist, dtype=float),
        "action": np.array(action_hist, dtype=float),
        "reward": np.array(reward_hist, dtype=float),
        "hold_time_sec": np.array(hold_time_hist, dtype=float),
        "params": env.params_num,
        "quad_config": {
            "model_name": env.quad_meta["model_name"],
            "spec_source_url": env.quad_meta["spec_source_url"],
            "frame_span_m": env.quad_meta["frame_span_m"],
            "quad_l": env.quad_config.quad_l,
            "hover_rpm": env.quad_config.hover_rpm,
            "rpm_delta_limit": env.quad_config.rpm_delta_limit,
            "max_rpm": env.quad_config.max_rpm,
            "kf": env.quad_config.kf,
            "km": env.quad_config.km,
        },
        "goal_mode": GOAL_MODE,
        "sim_t_start": env.sim_t_start,
        "sim_t_end": env.sim_t_end,
        "dt": env.dt,
        "episode_steps": env.episode_steps,
        "terminated": final_terminated,
        "truncated": final_truncated,
        "termination_reason": final_reason,
        "success_hold_time_required_sec": SUCCESS_HOLD_TIME_SEC,
        "terminate_on_success": TERMINATE_ON_SUCCESS,
    }
    return rollout


def save_rollout_npz(rollout, filename):
    np.savez_compressed(filename, **rollout)
    print("Saved rollout:", filename)



In [ ]:
class EpisodeRewardLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.timesteps = []
        self.episode_rewards = []

    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                self.timesteps.append(self.num_timesteps)
                self.episode_rewards.append(info["episode"]["r"])
        return True


class PeriodicRolloutSaver(BaseCallback):
    def __init__(self, eval_env, save_dir, algo_name, every_timesteps, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.save_dir = Path(save_dir)
        self.algo_name = algo_name
        self.every_timesteps = int(every_timesteps)
        self.next_save = self.every_timesteps

    def _on_step(self):
        if (not SAVE_PERIODIC_ROLLOUTS) or self.num_timesteps < self.next_save:
            return True

        rollout = run_policy_rollout(self.model, self.eval_env, deterministic=True)
        rollout_filename = self.save_dir / f"{self.algo_name.lower()}_rollout_step_{self.num_timesteps:08d}.npz"
        save_rollout_npz(rollout, rollout_filename)
        self.next_save += self.every_timesteps
        return True


## Environment / Model Builders

These helper functions keep the training loop short and make it easy to compare algorithms.


In [ ]:
def make_env(seed_offset=0):
    def _factory():
        return Monitor(ThreeDOFQuadRLEnv(seed=SEED + seed_offset))
    return _factory


def build_model(algo_name, env):
    algo_name = algo_name.upper()

    if algo_name == "PPO":
        return PPO("MlpPolicy", env, seed=SEED, **PPO_KWARGS)

    if algo_name == "SAC":
        return SAC("MlpPolicy", env, seed=SEED, **SAC_KWARGS)

    if algo_name == "TD3":
        action_noise = NormalActionNoise(mean=np.zeros(4), sigma=0.15 * np.ones(4))
        return TD3("MlpPolicy", env, seed=SEED, action_noise=action_noise, **TD3_KWARGS)

    raise ValueError(f"Unsupported algorithm: {algo_name}")


## Train Algorithms

This loop trains all algorithms listed in `ALGORITHMS_TO_TRAIN`, saves the controller, runs one deterministic evaluation rollout, and stores reward logs for comparison plots.


In [ ]:
training_summaries = {}
trained_model_paths = {}
final_rollout_paths = {}

for algo_name in ALGORITHMS_TO_TRAIN:
    algo_name = algo_name.upper()
    print("\n" + "=" * 72)
    print(f"Training {algo_name}")
    print("=" * 72)

    algo_model_dir = MODEL_DIR / algo_name.lower()
    algo_ckpt_dir = CHECKPOINT_DIR / algo_name.lower()
    algo_rollout_dir = ROLLOUT_DIR / algo_name.lower()
    algo_log_dir = LOG_DIR / algo_name.lower()

    for directory in [algo_model_dir, algo_ckpt_dir, algo_rollout_dir, algo_log_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    train_env = DummyVecEnv([make_env(seed_offset=0)])
    eval_env = DummyVecEnv([make_env(seed_offset=1000)])
    eval_rollout_env = ThreeDOFQuadRLEnv(seed=SEED + 2000)

    model = build_model(algo_name, train_env)
    reward_logger = EpisodeRewardLoggerCallback()

    callbacks = [reward_logger]

    if SAVE_CHECKPOINTS:
        callbacks.append(
            CheckpointCallback(
                save_freq=CHECKPOINT_EVERY,
                save_path=str(algo_ckpt_dir),
                name_prefix=f"{algo_name.lower()}_3dof_rig",
            )
        )

    callbacks.append(
        EvalCallback(
            eval_env,
            best_model_save_path=str(algo_model_dir / "best_model"),
            log_path=str(algo_log_dir),
            eval_freq=EVAL_EVERY,
            deterministic=True,
            render=False,
        )
    )

    callbacks.append(
        PeriodicRolloutSaver(
            eval_env=eval_rollout_env,
            save_dir=algo_rollout_dir,
            algo_name=algo_name,
            every_timesteps=ROLLOUT_SAVE_EVERY,
        )
    )

    total_steps = int(TRAIN_TIMESTEPS[algo_name])
    model.learn(total_timesteps=total_steps, callback=callbacks, progress_bar=False)

    final_model_path = algo_model_dir / f"{algo_name.lower()}_3dof_rig_final"
    model.save(str(final_model_path))
    trained_model_paths[algo_name] = final_model_path.with_suffix(".zip")
    print("Saved trained model to", trained_model_paths[algo_name])

    final_rollout = run_policy_rollout(model, ThreeDOFQuadRLEnv(seed=SEED + 3000), deterministic=True)
    if SAVE_FINAL_ROLLOUT:
        final_rollout_path = algo_rollout_dir / f"{algo_name.lower()}_rollout_final.npz"
        save_rollout_npz(final_rollout, final_rollout_path)
        final_rollout_paths[algo_name] = final_rollout_path

    eval_log_file = algo_log_dir / "evaluations.npz"
    eval_log = np.load(eval_log_file, allow_pickle=True) if eval_log_file.exists() else None

    summary = {
        "episode_reward_timesteps": np.array(reward_logger.timesteps, dtype=float),
        "episode_rewards": np.array(reward_logger.episode_rewards, dtype=float),
        "eval_timesteps": np.array(eval_log["timesteps"], dtype=float).reshape(-1) if eval_log is not None else np.array([]),
        "eval_mean_rewards": np.mean(eval_log["results"], axis=1) if eval_log is not None else np.array([]),
        "final_rollout": final_rollout,
    }
    training_summaries[algo_name] = summary

    if SAVE_TRAINING_HISTORY:
        np.savez_compressed(
            algo_log_dir / f"{algo_name.lower()}_training_history.npz",
            episode_reward_timesteps=summary["episode_reward_timesteps"],
            episode_rewards=summary["episode_rewards"],
            eval_timesteps=summary["eval_timesteps"],
            eval_mean_rewards=summary["eval_mean_rewards"],
        )



## Reward-vs-Iteration Plots

Two views are helpful:
- episode reward as logged by the monitor wrapper
- periodic evaluation mean reward from the evaluation callback


In [ ]:
plt.figure(figsize=(10, 4.5))
for algo_name, summary in training_summaries.items():
    if len(summary["episode_reward_timesteps"]) == 0:
        continue
    plt.plot(summary["episode_reward_timesteps"], summary["episode_rewards"], linewidth=1.4, label=f"{algo_name} episode reward")
plt.grid(True)
plt.xlabel("Training timestep")
plt.ylabel("Episode reward")
plt.title("Episode Reward vs Training Iteration")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4.5))
for algo_name, summary in training_summaries.items():
    if len(summary["eval_timesteps"]) == 0:
        continue
    plt.plot(summary["eval_timesteps"], summary["eval_mean_rewards"], linewidth=2.0, marker="o", label=f"{algo_name} eval mean reward")
plt.grid(True)
plt.xlabel("Training timestep")
plt.ylabel("Evaluation mean reward")
plt.title("Evaluation Reward vs Training Iteration")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
labels = ["n1", "n2", "n3"]

for i, label in enumerate(labels):
    for algo_name, summary in training_summaries.items():
        rollout = summary["final_rollout"]
        ax[i].plot(rollout["t"], np.rad2deg(rollout["n"][:, i]), linewidth=1.8, label=f"{algo_name} {label}")
    desired_hist = next(iter(training_summaries.values()))["final_rollout"]["desired_state_history"]
    ax[i].plot(next(iter(training_summaries.values()))["final_rollout"]["t"], np.rad2deg(desired_hist[:, i]), "k--", linewidth=1.6, label=f"desired {label}")
    ax[i].set_ylabel(f"{label} [deg]")
    ax[i].grid(True)
    ax[i].legend(loc="best")

for algo_name, summary in training_summaries.items():
    rollout = summary["final_rollout"]
    ax[3].plot(rollout["t"], rollout["hold_time_sec"], linewidth=1.8, label=f"{algo_name} hold time")
ax[3].axhline(SUCCESS_HOLD_TIME_SEC, color="k", linestyle="--", label="required hold time")
ax[3].set_ylabel("hold [s]")
ax[3].set_xlabel("Time [s]")
ax[3].grid(True)
ax[3].legend(loc="best")

fig.suptitle("Final Deterministic Tracking Comparison")
plt.tight_layout()
plt.show()



In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
labels = ["n1", "n2", "n3"]

for i, label in enumerate(labels):
    for algo_name, summary in training_summaries.items():
        rollout = summary["final_rollout"]
        ax[i].plot(rollout["t"], np.rad2deg(rollout["n"][:, i]), linewidth=1.8, label=f"{algo_name} {label}")
    desired_hist = next(iter(training_summaries.values()))["final_rollout"]["desired_state_history"]
    ax[i].plot(next(iter(training_summaries.values()))["final_rollout"]["t"], np.rad2deg(desired_hist[:, i]), "k--", linewidth=1.6, label=f"desired {label}")
    ax[i].set_ylabel(f"{label} [deg]")
    ax[i].grid(True)
    ax[i].legend(loc="best")

ax[-1].set_xlabel("Time [s]")
fig.suptitle("Final Deterministic Tracking Comparison")
plt.tight_layout()
plt.show()


## Save And Reload A Trained Agent

Use `VISUALIZE_ALGO` to choose which trained algorithm to reload for later analysis.


In [ ]:
selected_model_path = trained_model_paths[VISUALIZE_ALGO.upper()]
print("Selected model path:", selected_model_path)

if VISUALIZE_ALGO.upper() == "PPO":
    reloaded_model = PPO.load(str(selected_model_path))
elif VISUALIZE_ALGO.upper() == "SAC":
    reloaded_model = SAC.load(str(selected_model_path))
elif VISUALIZE_ALGO.upper() == "TD3":
    reloaded_model = TD3.load(str(selected_model_path))
else:
    raise ValueError(f"Unsupported VISUALIZE_ALGO: {VISUALIZE_ALGO}")

print("Reloaded model for", VISUALIZE_ALGO)


## Choose Which Iteration To Visualize

You asked whether the animation is always the last iteration.

Answer:
- no, it does **not** have to be the last one
- you can animate the final rollout
- or choose a saved periodic rollout by timestep
- or pass a full file path manually


In [ ]:
def resolve_rollout_file(algo_name, selector):
    algo_name = algo_name.upper()
    algo_rollout_dir = ROLLOUT_DIR / algo_name.lower()

    if isinstance(selector, str) and selector == "final":
        return algo_rollout_dir / f"{algo_name.lower()}_rollout_final.npz"

    if isinstance(selector, (int, np.integer)):
        return algo_rollout_dir / f"{algo_name.lower()}_rollout_step_{int(selector):08d}.npz"

    if isinstance(selector, str):
        return Path(selector)

    raise ValueError(f"Unsupported rollout selector: {selector}")


ROLLOUT_TO_ANIMATE = resolve_rollout_file(VISUALIZE_ALGO, VISUALIZE_ROLLOUT_SELECTOR)
print("Rollout selected for animation:", ROLLOUT_TO_ANIMATE)


rl_rollout = np.load(ROLLOUT_TO_ANIMATE, allow_pickle=True)
rl_t = rl_rollout["t"]
rl_n = rl_rollout["n"]
rl_nd = rl_rollout["nd"]
rl_states = np.hstack([rl_n, rl_nd])
rl_desired_state_history = rl_rollout["desired_state_history"]

print("Loaded rollout for animation:", ROLLOUT_TO_ANIMATE)
print("State trajectory shape:", rl_states.shape)
print("Desired trajectory shape:", rl_desired_state_history.shape)
if "sim_t_end" in rl_rollout.files:
    print("Saved rollout sim_t_end [s]:", float(rl_rollout["sim_t_end"]))
if "terminated" in rl_rollout.files:
    print("terminated:", bool(rl_rollout["terminated"]))
if "truncated" in rl_rollout.files:
    print("truncated:", bool(rl_rollout["truncated"]))
if "termination_reason" in rl_rollout.files:
    print("termination_reason:", rl_rollout["termination_reason"]) 



In [ ]:
rl_rollout = np.load(ROLLOUT_TO_ANIMATE, allow_pickle=True)
rl_t = rl_rollout["t"]
rl_n = rl_rollout["n"]
rl_nd = rl_rollout["nd"]
rl_states = np.hstack([rl_n, rl_nd])
rl_desired_state_history = rl_rollout["desired_state_history"]

print("Loaded rollout for animation:", ROLLOUT_TO_ANIMATE)
print("State trajectory shape:", rl_states.shape)
print("Desired trajectory shape:", rl_desired_state_history.shape)


## Animation Geometry Helpers

The target point in inertial coordinates is computed by taking the UAV-body point `rB_A` in the arm frame and rotating it using the **desired** joint angles at each frame.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# These geometry lengths must stay synchronized with arm_geometry_parameters.ipynb.
mm = 1e-3
L_C_to_bend = 300.0 * mm
L_diagonal = 172.75416 * mm
L_bottom = 60.0 * mm
L_upright = 162.0 * mm

params_num = get_params()
rC_A = np.array([params_num['xC_A'], params_num['yC_A'], params_num['zC_A']], dtype=float)
rB_A = np.array([params_num['xB_A'], params_num['yB_A'], params_num['zB_A']], dtype=float)
rO_A = np.array([params_num['xO_A'], params_num['yO_A'], params_num['zO_A']], dtype=float)

C_to_OA = -rC_A[0]
diagonal_x_run = float(np.sqrt(max(L_diagonal**2 - L_upright**2, 0.0)))

pC_ref = np.array([0.0, 0.0, 0.0])
p_bend_ref = np.array([L_C_to_bend, 0.0, 0.0])
p_lower_ref = p_bend_ref + np.array([diagonal_x_run, 0.0, -L_upright])
p_bottom_ref = p_lower_ref + np.array([L_bottom, 0.0, 0.0])
pB_ref = p_bottom_ref + np.array([0.0, 0.0, L_upright])
pOA_ref = np.array([C_to_OA, 0.0, 0.0])
arm_nodes_A = np.vstack([pC_ref, p_bend_ref, p_lower_ref, p_bottom_ref, pB_ref]) - pOA_ref

m_total = params_num['ma'] + params_num['mb'] + params_num['mc']
rG_A = (
    params_num['ma'] * rO_A
    + params_num['mb'] * rB_A
    + params_num['mc'] * rC_A
) / m_total


def C_AN_numeric(n1, n2, n3):
    c1, s1 = np.cos(n1), np.sin(n1)
    c2, s2 = np.cos(n2), np.sin(n2)
    c3, s3 = np.cos(n3), np.sin(n3)

    R1_n3 = np.array([
        [1.0, 0.0, 0.0],
        [0.0, c3, s3],
        [0.0, -s3, c3],
    ])
    R2_n2 = np.array([
        [c2, 0.0, -s2],
        [0.0, 1.0, 0.0],
        [s2, 0.0, c2],
    ])
    R3_n1 = np.array([
        [c1, s1, 0.0],
        [-s1, c1, 0.0],
        [0.0, 0.0, 1.0],
    ])
    return R1_n3 @ R2_n2 @ R3_n1


def points_A_to_N(points_A, state):
    C_AN = C_AN_numeric(state[0], state[1], state[2])
    return (C_AN.T @ np.asarray(points_A, dtype=float).T).T


In [ ]:
# Precompute point clouds for actual and target states so the axes remain stable.
all_points_N = []
for state, desired_state in zip(rl_states, rl_desired_state_history):
    actual_points_A = np.vstack([arm_nodes_A, rB_A, rC_A, rO_A, rG_A, np.zeros(3)])
    desired_body_A = np.vstack([rB_A])
    all_points_N.append(points_A_to_N(actual_points_A, state))
    all_points_N.append(points_A_to_N(desired_body_A, desired_state))
all_points_N = np.vstack(all_points_N)

mins = all_points_N.min(axis=0)
maxs = all_points_N.max(axis=0)
center = 0.5 * (mins + maxs)
span = float(np.max(maxs - mins))
padding = max(0.08 * span, 0.05)
half_range = 0.5 * span + padding


In [ ]:
fig = plt.figure(figsize=(7.5, 6.5))
ax = fig.add_subplot(111, projection='3d')
ax.set_xlim(center[0] - half_range, center[0] + half_range)
ax.set_ylim(center[1] - half_range, center[1] + half_range)
ax.set_zlim(center[2] - half_range, center[2] + half_range)
ax.set_box_aspect((1, 1, 1))
ax.set_xlabel('X_N [m]')
ax.set_ylabel('Y_N [m]')
ax.set_zlabel('Z_N [m]')
ax.view_init(elev=22, azim=-45)

arm_line, = ax.plot([], [], [], color='black', linewidth=3.0, marker='o', markersize=3.5, label='arm')
pivot_point, = ax.plot([], [], [], marker='x', color='crimson', markersize=9, linestyle='None', label='pivot')
body_point, = ax.plot([], [], [], marker='o', color='darkorange', markeredgecolor='black', markersize=8, linestyle='None', label='actual body')
comp_point, = ax.plot([], [], [], marker='o', color='darkblue', markeredgecolor='black', markersize=8, linestyle='None', label='compensator')
arm_com_point, = ax.plot([], [], [], marker='o', color='purple', markeredgecolor='black', markersize=7, linestyle='None', label='arm COM')
total_com_point, = ax.plot([], [], [], marker='o', color='limegreen', markeredgecolor='black', markersize=7, linestyle='None', label='total COM')
target_body_point, = ax.plot([], [], [], marker='*', color='red', markeredgecolor='black', markersize=12, linestyle='None', label='target body point')
ax.legend(loc='upper left', framealpha=1.0)


def set_point(plot_obj, point):
    plot_obj.set_data([point[0]], [point[1]])
    plot_obj.set_3d_properties([point[2]])


def init_anim():
    arm_line.set_data([], [])
    arm_line.set_3d_properties([])
    return arm_line, pivot_point, body_point, comp_point, arm_com_point, total_com_point, target_body_point


def update_anim(k):
    state = rl_states[k]
    desired_state = rl_desired_state_history[k]

    nodes_N = points_A_to_N(arm_nodes_A, state)
    rB_N, rC_N, rO_N, rG_N, OA_N = points_A_to_N(
        np.vstack([rB_A, rC_A, rO_A, rG_A, np.zeros(3)]),
        state,
    )
    desired_body_N = points_A_to_N(np.vstack([rB_A]), desired_state)[0]

    arm_line.set_data(nodes_N[:, 0], nodes_N[:, 1])
    arm_line.set_3d_properties(nodes_N[:, 2])

    set_point(pivot_point, OA_N)
    set_point(body_point, rB_N)
    set_point(comp_point, rC_N)
    set_point(arm_com_point, rO_N)
    set_point(total_com_point, rG_N)
    set_point(target_body_point, desired_body_N)

    ax.set_title(
        f't={rl_t[k]:.2f}s | '
        f'actual [deg]=({np.degrees(state[0]):+.1f}, {np.degrees(state[1]):+.1f}, {np.degrees(state[2]):+.1f}) | '
        f'desired [deg]=({np.degrees(desired_state[0]):+.1f}, {np.degrees(desired_state[1]):+.1f}, {np.degrees(desired_state[2]):+.1f})'
    )

    return arm_line, pivot_point, body_point, comp_point, arm_com_point, total_com_point, target_body_point

ani = FuncAnimation(
    fig,
    update_anim,
    frames=len(rl_states),
    init_func=init_anim,
    interval=1000 * DT,
    blit=False,
    repeat=False,
)

plt.close(fig)
HTML(ani.to_html5_video())
